In [ ]:
# 向量数据库

In [1]:
from langchain.document_loaders import TextLoader
raw_documents = TextLoader('./data/sora.txt', encoding="utf-8").load()

In [2]:
raw_documents

[Document(metadata={'source': './data/sora.txt'}, page_content='一、Sora是什么？为什么说Sora是人工智能划时代的突破?\n\n2024年2月16日凌晨，OpenAI在官网发布了最新的文生视频模型Sora。Sora不仅突破了现有文生视频模型10秒左右的连贯性局限，而且展示出了更精细的画质、多角度多镜头切换中保持一致性等能力。最重要的是，Sora较好地表现出了现实世界中的逻辑，比如在模型生成的两艘海盗船在咖啡杯内航行的视频中，咖啡的流动完全符合现实世界中的流体力学；比如一则宠物猫等待主人起床的视频中，宠物猫踩奶的动作、对主人鼻头的轻触都符合现实世界中动物的习性。\n\n尽管出于安全性和测试验证等原因，Sora尚未开放给公众使用，但通过观察OpenAI发布的demo，不难发现Sora已经初步具备了理解现实世界运行规律的能力，即“涌现”的能力被成功得从大语言模型复制到了文生视频模型中。假以时日，人类或将很难区分现实世界和由AI生成的虚拟世界。除了为视频制作、电影剪辑、游戏设计等工作提升效率外，一个完全由数据驱动的数字世界或许已经近在咫尺。英伟达人工智能研究院首席科学家JimFan称“这是视频生成领域的GPT-3时刻”。360董事长周鸿祎称“Sora意味着AGI（通用人工智能）实现将从10年缩短到1年”。\n\n二、Sora是如何实现的？\n\n尽管OpenAI在Sora的技术文档中并未公开所有模型细节，但我们可以通过其描述大致推测出，Sora的实现仍然依赖于OpenAI在大语言模型领域取得巨大成功的“大力出奇迹”思想，即通过大幅提升训练数据和参数规模实现视频精度和对现实世界物理关系的“涌现”。\n\n相比GPT模型，Sora的成功之处在于为图像视频等多模态数据找到了适合Transformer架构的表征方式，从而将ScalingLaw从语言模型复制到了图像视频模型。此外，Sora在训练时还借助了DALL·E3生成的高质量文本描述，在推理时借助了GPT对用户输入进行扩展，可谓“站在巨人肩膀上”更进一步。\n\n三、Sora对AI应用和算力需求带来哪些影响?\n\n对于应用而言，Sora生成的视频已经达到了大部分消费级场景的使用要求，将为短视频创作等创意产业带来繁荣。随着模型升级，预计也将对电影、游戏等行业的制

In [3]:
# 接下来，通过文档切割器`RecursiveCharacterTextSplitter`,将上面完整的Docement对象切分为多个chunks。

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], # 默认
    chunk_size=500, #块长度
    chunk_overlap=20, #重叠字符串长度
    add_start_index=True
)
documents = text_splitter.split_documents(raw_documents)

In [6]:
# 准备向量模型，这里我们依然使用BaiChuan。

In [8]:
from langchain_community.embeddings import BaichuanTextEmbeddings
import os

# 设置API密钥
key = open('./ken_files/baichuan_API-Key.md').read().strip()
embeddings_model = BaichuanTextEmbeddings(api_key=key)

In [9]:
#创建 Chroma 数据库实例

In [10]:
from langchain_community.vectorstores import Chroma
#documents:文档将被转换为向量并存储在数据库中
#embeddings_model:向量的嵌入模型
#persist_directory:如果指定路径，向量存储将被持久化到此目录。如果未指定，数据将只在内存中临时存在。
db = Chroma.from_documents(documents, embeddings_model)

In [11]:
#使用向量数据库（`db`）来查找与查询语句 `query` 相似的文档

In [13]:
query = "什么是Sora"
#在数据库中进行相似性搜索
#通过关键词k，可以设置返回多少个在查询过程中与Query最接近的Chunks
docs = db.similarity_search(query,k=2)
print(docs[0].page_content)

一、Sora是什么？为什么说Sora是人工智能划时代的突破?

2024年2月16日凌晨，OpenAI在官网发布了最新的文生视频模型Sora。Sora不仅突破了现有文生视频模型10秒左右的连贯性局限，而且展示出了更精细的画质、多角度多镜头切换中保持一致性等能力。最重要的是，Sora较好地表现出了现实世界中的逻辑，比如在模型生成的两艘海盗船在咖啡杯内航行的视频中，咖啡的流动完全符合现实世界中的流体力学；比如一则宠物猫等待主人起床的视频中，宠物猫踩奶的动作、对主人鼻头的轻触都符合现实世界中动物的习性。


In [14]:
query = "Sora在训练时消耗了多少算力？"
docs = db.similarity_search(query)
print(docs[0].page_content)

对于算力而言，由于OpenAI并未公布模型架构的细节，很难推测训练Sora具体消耗了多少算力，但既然ScalingLaw，或者说“大力出奇迹”依然是当前AI模型实现“涌现”的黄金法则，就意味着对算力的需求仍然没有看到“拐点”。如果Sora的训练确实使用了合成数据，意味着可供训练的数据远未耗尽，人类对AI模型参数和训练数据的提升还远远没有达到瓶颈。甚至随着AI模型合成数据能力的增强，模型推理结果本身就可以作为训练的一部分，从而实现AI的自我迭代。此外，视频生成推理需要更大的VRAM或带宽，如果Sora开放使用后如期推动各类视频创作的繁荣，当前电信和数通网络的带宽都需要大幅升级。作为广义算力的一部分，网络设备的需求也将爆发式增长。

对于投资而言，Sora最大的意义在于证明了AI产业的创新浪潮还远未停歇。Sora的“前辈”ChatGPT发布以来，芯片龙头英伟达、博通股价分别上涨超300%、130%，软件应用龙头微软上涨超60%。Sora作为多模态大模型，向公众开放后预计对算力需求更大、对软件应用成长空间提升更显著，有望进一步提升相关产业投资价值。


In [15]:
#在上⼀个示例的基础上，如果您想要保存到磁盘，只需初始化 Chroma 客户端并传递要保存数据的目录。

In [21]:
# 保存到磁盘
db2 = Chroma.from_documents(documents,embeddings_model,persist_directory="./chroma_db")
docs = db2.similarity_search(query)

In [26]:
# 从磁盘加载
db3 = Chroma(persist_directory="./chroma_db", embedding_function=embeddings_model)
docs = db3.similarity_search(query)
print(docs[0].page_content)

对于算力而言，由于OpenAI并未公布模型架构的细节，很难推测训练Sora具体消耗了多少算力，但既然ScalingLaw，或者说“大力出奇迹”依然是当前AI模型实现“涌现”的黄金法则，就意味着对算力的需求仍然没有看到“拐点”。如果Sora的训练确实使用了合成数据，意味着可供训练的数据远未耗尽，人类对AI模型参数和训练数据的提升还远远没有达到瓶颈。甚至随着AI模型合成数据能力的增强，模型推理结果本身就可以作为训练的一部分，从而实现AI的自我迭代。此外，视频生成推理需要更大的VRAM或带宽，如果Sora开放使用后如期推动各类视频创作的繁荣，当前电信和数通网络的带宽都需要大幅升级。作为广义算力的一部分，网络设备的需求也将爆发式增长。

对于投资而言，Sora最大的意义在于证明了AI产业的创新浪潮还远未停歇。Sora的“前辈”ChatGPT发布以来，芯片龙头英伟达、博通股价分别上涨超300%、130%，软件应用龙头微软上涨超60%。Sora作为多模态大模型，向公众开放后预计对算力需求更大、对软件应用成长空间提升更显著，有望进一步提升相关产业投资价值。


In [27]:
#在构建实际应用程序时，除了添加和检索，非常多的情况下还需要更新和删除数据，
#这就需要借助到Chroma类定义的` ids` 参数，它可以传入文件名或任意的标识。
#我们需要先根据分成Chunks构建起唯一的对应id。

In [28]:
import uuid
ids = [str(uuid.uuid4()) for _ in documents]
new_db = Chroma.from_documents(documents, embeddings_model,ids=ids)

In [29]:
#接着，执行`update_document`方法进行更新，如下所示：

In [36]:
new_db.update_document(ids[0], docs[0])

In [37]:
#与任何其他数据库一样，在向量数据库中，也可以使用`.add`、`.get` 、`.update` `.delete`等方法，
# 但如果想直接访问，需要执行`._collection.method()`。所以我们可以通过如下的代码形式，查看更新后的内容：

In [38]:
print(new_db._collection.get(ids=[ids[0]]))

{'ids': ['a391f381-ddf7-425a-a26b-6a488ff93165'], 'embeddings': None, 'documents': ['对于算力而言，由于OpenAI并未公布模型架构的细节，很难推测训练Sora具体消耗了多少算力，但既然ScalingLaw，或者说“大力出奇迹”依然是当前AI模型实现“涌现”的黄金法则，就意味着对算力的需求仍然没有看到“拐点”。如果Sora的训练确实使用了合成数据，意味着可供训练的数据远未耗尽，人类对AI模型参数和训练数据的提升还远远没有达到瓶颈。甚至随着AI模型合成数据能力的增强，模型推理结果本身就可以作为训练的一部分，从而实现AI的自我迭代。此外，视频生成推理需要更大的VRAM或带宽，如果Sora开放使用后如期推动各类视频创作的繁荣，当前电信和数通网络的带宽都需要大幅升级。作为广义算力的一部分，网络设备的需求也将爆发式增长。\n\n对于投资而言，Sora最大的意义在于证明了AI产业的创新浪潮还远未停歇。Sora的“前辈”ChatGPT发布以来，芯片龙头英伟达、博通股价分别上涨超300%、130%，软件应用龙头微软上涨超60%。Sora作为多模态大模型，向公众开放后预计对算力需求更大、对软件应用成长空间提升更显著，有望进一步提升相关产业投资价值。'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'start_index': 1047, 'source': './data/sora.txt'}]}


In [39]:
# 当然，也可以直接进行删除操作，在删除之前，先看一下有多少个Chunks，代码如下所示：

In [40]:
print(new_db._collection.count())

12


In [41]:
#删除最后一个chunk

In [42]:
new_db._collection.delete(ids=[ids[-1]])

In [43]:
#再次查看存储的总Chunks数

In [44]:
print(new_db._collection.count())

11


In [ ]:
#**拓展：MMR**

In [45]:
#MMR（Maximal Marginal Relevance，最大边际相关性）是一种信息检索和文本摘要技术，
#用于在选择文档或文本片段时平衡相关性和多样性。其主要目的是在检索结果中既包含与查询高度相关的内容，
#又避免结果之间的高度冗余。因此MMR的作用就是：

In [46]:
#- 计算相关性：首先，计算每个候选文档与查询的相似性得分。
#- 计算多样性：然后，计算每个候选文档与已选文档集合的相似性得分。
#- 选择文档：在每一步选择一个文档，使得该文档在相关性和多样性之间达到最佳平衡。

In [47]:
retriever = db.as_retriever(search_type="mmr")
retriever.invoke(query)[0]

Document(metadata={'start_index': 1047, 'source': './data/sora.txt'}, page_content='对于算力而言，由于OpenAI并未公布模型架构的细节，很难推测训练Sora具体消耗了多少算力，但既然ScalingLaw，或者说“大力出奇迹”依然是当前AI模型实现“涌现”的黄金法则，就意味着对算力的需求仍然没有看到“拐点”。如果Sora的训练确实使用了合成数据，意味着可供训练的数据远未耗尽，人类对AI模型参数和训练数据的提升还远远没有达到瓶颈。甚至随着AI模型合成数据能力的增强，模型推理结果本身就可以作为训练的一部分，从而实现AI的自我迭代。此外，视频生成推理需要更大的VRAM或带宽，如果Sora开放使用后如期推动各类视频创作的繁荣，当前电信和数通网络的带宽都需要大幅升级。作为广义算力的一部分，网络设备的需求也将爆发式增长。\n\n对于投资而言，Sora最大的意义在于证明了AI产业的创新浪潮还远未停歇。Sora的“前辈”ChatGPT发布以来，芯片龙头英伟达、博通股价分别上涨超300%、130%，软件应用龙头微软上涨超60%。Sora作为多模态大模型，向公众开放后预计对算力需求更大、对软件应用成长空间提升更显著，有望进一步提升相关产业投资价值。')

In [ ]:
#7.7.2 Faiss的使用（拓展）

In [ ]:
# Faiss 是由 Facebook 团队开源的向量检索工具，专为高维空间的海量数据提供高效、可靠的相似性检索方案。
# Faiss 支持 Linux、macOS 和 Windows 操作系统，在处理百万级向量的相似性检索时，
# Faiss 可以在牺牲一定搜索准确度的情况下，实现小于 10ms 的响应时间。

In [48]:
# pip install -U faiss-cpu tiktoken

In [49]:
# 如果您想使用启用了 GPU 的版本，也可以安装 faiss-gpu 。

In [50]:
from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(docs, embeddings_model)
query = "Pixar公司是做什么的?"
docs = db.similarity_search(query)
print(docs[0].page_content)

对于算力而言，由于OpenAI并未公布模型架构的细节，很难推测训练Sora具体消耗了多少算力，但既然ScalingLaw，或者说“大力出奇迹”依然是当前AI模型实现“涌现”的黄金法则，就意味着对算力的需求仍然没有看到“拐点”。如果Sora的训练确实使用了合成数据，意味着可供训练的数据远未耗尽，人类对AI模型参数和训练数据的提升还远远没有达到瓶颈。甚至随着AI模型合成数据能力的增强，模型推理结果本身就可以作为训练的一部分，从而实现AI的自我迭代。此外，视频生成推理需要更大的VRAM或带宽，如果Sora开放使用后如期推动各类视频创作的繁荣，当前电信和数通网络的带宽都需要大幅升级。作为广义算力的一部分，网络设备的需求也将爆发式增长。

对于投资而言，Sora最大的意义在于证明了AI产业的创新浪潮还远未停歇。Sora的“前辈”ChatGPT发布以来，芯片龙头英伟达、博通股价分别上涨超300%、130%，软件应用龙头微软上涨超60%。Sora作为多模态大模型，向公众开放后预计对算力需求更大、对软件应用成长空间提升更显著，有望进一步提升相关产业投资价值。


In [51]:
#MMR使用：

In [52]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
print(docs[0].page_content)

对于算力而言，由于OpenAI并未公布模型架构的细节，很难推测训练Sora具体消耗了多少算力，但既然ScalingLaw，或者说“大力出奇迹”依然是当前AI模型实现“涌现”的黄金法则，就意味着对算力的需求仍然没有看到“拐点”。如果Sora的训练确实使用了合成数据，意味着可供训练的数据远未耗尽，人类对AI模型参数和训练数据的提升还远远没有达到瓶颈。甚至随着AI模型合成数据能力的增强，模型推理结果本身就可以作为训练的一部分，从而实现AI的自我迭代。此外，视频生成推理需要更大的VRAM或带宽，如果Sora开放使用后如期推动各类视频创作的繁荣，当前电信和数通网络的带宽都需要大幅升级。作为广义算力的一部分，网络设备的需求也将爆发式增长。

对于投资而言，Sora最大的意义在于证明了AI产业的创新浪潮还远未停歇。Sora的“前辈”ChatGPT发布以来，芯片龙头英伟达、博通股价分别上涨超300%、130%，软件应用龙头微软上涨超60%。Sora作为多模态大模型，向公众开放后预计对算力需求更大、对软件应用成长空间提升更显著，有望进一步提升相关产业投资价值。


In [53]:
#您还可以保存和加载 FAISS 索引。这样做很有用，因为您不必每次使用时都重新创建它。

In [55]:
#保存索引
db.save_local("faiss_index")
#读取索引
new_db = FAISS.load_local("faiss_index", embeddings_model,allow_dangerous_deserialization=True)
#进行检索
docs = new_db.similarity_search(query)

In [56]:
#**Faiss与Chroma使用场景的区别**
#1. 数据规模和性能需求：
#   - Faiss：更适合处理大规模数据，尤其是在需要利用GPU加速来提高搜索性能的场景下表现出色。例如在处理海量的图像特征向量、大规模的文本嵌入向量等场景中，Faiss能够快速地进行相似性搜索，满足对实时性和高性能的要求。
#   - Chroma：适用于中小规模数据或对性能要求不是特别极致的场景。虽然Chroma也具有一定的性能优化，但在处理超大规模数据时，其性能可能受限于硬件资源，不过对于一般的小型项目或原型开发来说已经足够。
#
#2. 开发和集成难度：
#
#   - Faiss：需要开发者对向量检索算法和索引结构有一定的了解，手动管理索引的创建、训练和持久化等操作，
#      开发和集成难度相对较大。但它的灵活性也使得在一些特定场景下可以根据需求进行深度定制。
#   - Chroma：提供了更简单的API和更便捷的使用方式，开箱即用，类似于一个完整的数据库，
#      对于开发者来说更容易上手和使用，能够快速集成到各种应用中，
#      特别适合快速原型开发和那些对数据库内部细节不太关注的应用场景。